# Problem 41 · Task 1 — Predicting `target01` (Regression)

**Goal:** build a regression model that predicts the continuous target `target01`
from ~272 anonymised numeric features, then generate predictions for an unlabeled
evaluation set.

### Approach at a glance
1. **Load & assemble** the feature matrix and targets.
2. **Split** into train / test sets (80 / 20).
3. **Explore** the data and **classify every feature by its statistical type**
   (binary, integer-count, bounded-`[0,1]`, continuous float).
4. **Select** the 150 most informative features using Random Forest importances.
5. **Train & compare** five regressors — XGBoost, CatBoost, LightGBM,
   Random Forest and Linear Regression.
6. **Evaluate** the chosen model on the hold-out set and write the submission
   file `EVAL_target01_41.csv`.

### Dataset
| File | Rows | Columns | Description |
|------|------|---------|-------------|
| `problem_41/dataset_41.csv` | 10,000 | 272 | Training features `feat_0 … feat_271` |
| `problem_41/target_41.csv`  | 10,000 | 2   | Labels `target01`, `target02` (only `target01` used here) |
| `problem_41/EVAL_41.csv`    | 10,000 | 272 | Unlabeled evaluation features |


## 1. Load & Assemble the Dataset

Load the feature matrix and the target file, then concatenate them so the targets travel with their rows. `target02` is dropped — it is the subject of Task 2 and is not a valid predictor for `target01`.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [17]:
%matplotlib inline

In [18]:
df = pd.read_csv('problem_41/dataset_41.csv')

In [19]:
df_targets = pd.read_csv("problem_41/target_41.csv")

In [20]:
df.columns

Index(['feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6',
       'feat_7', 'feat_8', 'feat_9',
       ...
       'feat_263', 'feat_264', 'feat_265', 'feat_266', 'feat_267', 'feat_268',
       'feat_269', 'feat_270', 'feat_271', 'feat_272'],
      dtype='object', length=273)

In [21]:
df = pd.concat([df, df_targets], axis=1)

In [22]:
df

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272,target01,target02
0,0.060429,0.163290,46.0,-0.955850,0.202933,-0.617414,-0.964104,-1.582704,-1.003891,0.564653,...,-1.709070,0.510682,2.260921,1.738723,0.569321,-1.968904,0.080735,-0.854864,1.010363,-1.689414
1,0.948468,-1.939359,47.0,1.537635,1.865508,-1.845841,2.098390,-1.228306,1.923310,2.136063,...,1.040862,-0.076576,-2.645752,-1.222789,-0.762040,-0.291181,-0.361413,-1.260932,0.553083,-1.319422
2,0.959227,-0.696984,46.0,-0.560277,3.469353,-0.589864,-1.237769,2.949433,3.006876,-0.532553,...,1.928491,-2.033318,-3.709119,-0.837182,1.533737,0.821797,3.774493,-0.628873,0.299941,1.689612
3,0.682647,1.538574,9.0,1.179725,4.047743,3.568153,-1.050392,1.646529,-0.188768,-3.765864,...,-3.690466,-3.548139,-4.259292,-1.133941,0.966617,-1.500147,-0.455414,0.883379,0.588469,-0.728385
4,0.866248,0.774515,24.0,-0.765221,-1.995221,4.293457,-0.820623,0.812994,-0.894214,1.096028,...,1.323746,-1.074247,-3.016816,1.845526,1.134800,-1.114377,5.378954,1.765716,1.200542,2.119740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.731128,1.954098,0.0,0.723278,1.345687,3.019473,3.314710,0.444943,1.468696,-0.424910,...,-0.903109,3.260026,-1.028673,0.502366,-1.735416,1.446296,-0.383182,3.520398,0.499682,0.627622
9996,0.956628,5.067817,7.0,-0.322515,0.744496,3.433568,-0.980875,-1.016251,0.015362,-0.787093,...,-1.907276,-2.188096,0.213607,3.604545,-1.425140,0.258239,1.032970,0.678686,0.580556,0.283985
9997,0.008191,-0.324372,26.0,-0.410032,0.224640,-1.713738,1.222854,0.346244,0.289790,-2.067661,...,-1.023405,-1.424953,-0.048581,0.364134,-0.121381,-0.989501,0.455992,-0.293809,0.527039,-0.426513
9998,0.489696,0.655308,13.0,-1.521950,-1.504141,1.702731,3.133294,-2.067412,3.279543,1.039625,...,-1.165410,1.807009,3.190512,0.638915,2.263185,3.922717,-2.123733,0.010575,0.916117,1.699520


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 275 entries, feat_0 to target02
dtypes: float64(275)
memory usage: 21.0 MB


In [24]:
df.describe()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272,target01,target02
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,0.503104,0.480568,24.557600,0.219797,0.283793,0.246611,0.229681,0.356720,0.211238,0.535693,...,0.521536,0.235142,0.236095,0.236274,0.232304,0.248776,0.242132,0.272928,0.721163,-0.153366
std,0.289650,2.005855,14.483883,2.027269,2.026322,1.990801,2.001488,2.022389,2.008727,2.037439,...,2.041468,2.006722,2.030395,1.998245,1.995314,2.024212,2.008683,2.000034,0.237757,1.258333
min,0.000028,-7.314991,0.000000,-8.177571,-7.229206,-8.287052,-7.062289,-7.003741,-9.373280,-7.425622,...,-8.303189,-7.804743,-8.551021,-7.836784,-6.862910,-7.220598,-6.829705,-6.823869,0.291678,-2.844199
25%,0.251419,-0.876690,12.000000,-1.133938,-1.092048,-1.085532,-1.133705,-1.008447,-1.158991,-0.835244,...,-0.892925,-1.124195,-1.137190,-1.092732,-1.094269,-1.107575,-1.105108,-1.081963,0.540038,-1.143531
50%,0.507506,0.481427,25.000000,0.219892,0.300151,0.221894,0.257708,0.336163,0.217268,0.552220,...,0.517552,0.219798,0.210121,0.241079,0.227540,0.230156,0.256182,0.242590,0.606926,-0.496640
75%,0.755419,1.839089,37.000000,1.566710,1.631281,1.565426,1.561082,1.727246,1.588809,1.910642,...,1.895334,1.569332,1.613906,1.591302,1.588663,1.609289,1.606712,1.618271,0.925856,0.881925
max,0.999933,9.184359,49.000000,7.677687,7.526780,8.396989,7.029126,8.363670,8.627701,8.536040,...,8.709665,7.527872,9.022507,8.904775,7.724149,9.544657,8.468460,7.716347,1.400254,3.025131


In [25]:
df = df.drop("target02", axis=1)

In [26]:
df.head(5)

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_264,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272,target01
0,0.060429,0.163290,46.0,-0.955850,0.202933,-0.617414,-0.964104,-1.582704,-1.003891,0.564653,...,0.113773,-1.709070,0.510682,2.260921,1.738723,0.569321,-1.968904,0.080735,-0.854864,1.010363
1,0.948468,-1.939359,47.0,1.537635,1.865508,-1.845841,2.098390,-1.228306,1.923310,2.136063,...,-3.485232,1.040862,-0.076576,-2.645752,-1.222789,-0.762040,-0.291181,-0.361413,-1.260932,0.553083
2,0.959227,-0.696984,46.0,-0.560277,3.469353,-0.589864,-1.237769,2.949433,3.006876,-0.532553,...,0.066607,1.928491,-2.033318,-3.709119,-0.837182,1.533737,0.821797,3.774493,-0.628873,0.299941
3,0.682647,1.538574,9.0,1.179725,4.047743,3.568153,-1.050392,1.646529,-0.188768,-3.765864,...,0.583765,-3.690466,-3.548139,-4.259292,-1.133941,0.966617,-1.500147,-0.455414,0.883379,0.588469
4,0.866248,0.774515,24.0,-0.765221,-1.995221,4.293457,-0.820623,0.812994,-0.894214,1.096028,...,2.607052,1.323746,-1.074247,-3.016816,1.845526,1.134800,-1.114377,5.378954,1.765716,1.200542


## 2. Train / Test Split

An 80 / 20 split with a fixed `random_state` keeps the hold-out set untouched during model development and makes every result reproducible.

In [27]:
xtrain, xtest, ytrain, ytest = train_test_split(df.drop('target01',axis=1), df['target01'], test_size=0.20, random_state=42)

In [28]:
xtrain.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8000 entries, 9254 to 7270
Columns: 273 entries, feat_0 to feat_272
dtypes: float64(273)
memory usage: 16.7 MB


In [29]:
xtest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2000 entries, 6252 to 6929
Columns: 273 entries, feat_0 to feat_272
dtypes: float64(273)
memory usage: 4.2 MB


## 3. Exploratory Data Analysis

A quick look at the summary statistics of the training features to understand their scales and spot anything unusual before modelling.

In [31]:
xtrain.describe()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_263,feat_264,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,...,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,0.502842,0.486005,24.525625,0.229403,0.278454,0.256897,0.241986,0.343415,0.196142,0.528728,...,0.505250,0.244390,0.518462,0.236534,0.228792,0.233961,0.232488,0.250832,0.236466,0.256648
std,0.288346,1.998603,14.470626,2.033664,2.024339,1.993647,1.988825,2.024241,2.014680,2.040732,...,0.500004,2.025570,2.049816,2.002862,2.035116,2.001365,1.994509,2.019057,1.995191,2.002310
min,0.000028,-7.314991,0.000000,-8.177571,-7.229206,-8.287052,-7.062289,-7.003741,-9.373280,-7.425622,...,0.000000,-7.175225,-6.556997,-7.804743,-8.551021,-6.956532,-6.862910,-7.220598,-6.829705,-6.823869
25%,0.254087,-0.870972,12.000000,-1.127274,-1.108936,-1.092264,-1.108796,-1.011872,-1.182997,-0.852024,...,0.000000,-1.106173,-0.897852,-1.120604,-1.139189,-1.099552,-1.089089,-1.115549,-1.092792,-1.101662
50%,0.506436,0.485748,25.000000,0.217411,0.288676,0.223869,0.269723,0.325933,0.204914,0.540800,...,1.000000,0.283201,0.504351,0.218268,0.200918,0.240356,0.235000,0.225997,0.257031,0.241655
75%,0.753506,1.844937,37.000000,1.585740,1.633288,1.574408,1.568037,1.710321,1.583170,1.893304,...,1.000000,1.589884,1.888507,1.572171,1.606305,1.586981,1.590676,1.611223,1.579794,1.597366
max,0.999933,9.184359,49.000000,7.677687,7.526780,8.396989,7.029126,8.363670,8.627701,8.536040,...,1.000000,9.193137,8.709665,7.527872,9.022507,8.610690,7.724149,9.544657,8.468460,7.716347


## 4. Feature Typing & Preprocessing

The features are anonymised, so we infer each column's *statistical* type directly from its values and bucket every column into one of four groups:

- **binary** — two distinct values in `{0, 1}`
- **integer** — whole-number counts
- **bounded [0,1]** — continuous values confined to the unit interval
- **float** — unconstrained continuous values

This typing drives the downstream feature selection and lets us reason about which transforms (if any) each group needs.

In [32]:
numeric_cols = xtrain.select_dtypes(include=['int64', 'float64']).columns.tolist()
binary_cols = [
    col for col in numeric_cols
    if xtrain[col].dropna().nunique() == 2
    and set(xtrain[col].dropna().unique()).issubset({0, 1})
]

integer_cols = [
    col for col in numeric_cols
    if col not in binary_cols
    and (xtrain[col].dropna() % 1 == 0).all() 
]

bounded_01_cols = [
    col for col in numeric_cols
    if col not in binary_cols
    and xtrain[col].dropna().min() >= 0
    and xtrain[col].dropna().max() <= 1
    and not (xtrain[col].dropna() % 1 == 0).all()
]

float_cols = [
    col for col in numeric_cols
    if col not in binary_cols
    and col not in integer_cols
    and col not in bounded_01_cols
]

In [33]:
bounded_01_cols

['feat_0',
 'feat_15',
 'feat_53',
 'feat_64',
 'feat_71',
 'feat_77',
 'feat_116',
 'feat_126',
 'feat_154',
 'feat_162',
 'feat_165',
 'feat_178',
 'feat_187',
 'feat_194',
 'feat_204',
 'feat_207',
 'feat_210',
 'feat_214',
 'feat_222',
 'feat_239']

In [34]:
xtrain[bounded_01_cols].head(5)

,feat_0,feat_15,feat_53,feat_64,feat_71,feat_77,feat_116,feat_126,feat_154,feat_162,feat_165,feat_178,feat_187,feat_194,feat_204,feat_207,feat_210,feat_214,feat_222,feat_239
9254,0.045918,0.788820,0.202391,0.333896,0.561493,0.567136,0.698933,0.351345,0.429886,0.809337,0.181416,0.368707,0.255466,0.386783,0.942616,0.072375,0.397478,0.949063,0.404381,0.841409
1561,0.468781,0.960552,0.770149,0.464078,0.263094,0.781946,0.132786,0.291754,0.834003,0.186456,0.497630,0.934363,0.474616,0.168685,0.866256,0.897284,0.957511,0.584193,0.291207,0.298044
1670,0.611241,0.477252,0.834195,0.400137,0.664107,0.474778,0.228399,0.604658,0.467427,0.745765,0.452974,0.604382,0.793171,0.516167,0.854954,0.516712,0.200931,0.535102,0.673329,0.429581
6087,0.422402,0.780548,0.102220,0.670588,0.094344,0.504133,0.383810,0.668073,0.674674,0.511133,0.896284,0.015021,0.085006,0.191439,0.234101,0.274742,0.051573,0.013760,0.733240,0.991190
6669,0.383690,0.891060,0.697954,0.071554,0.698745,0.710876,0.310143,0.699038,0.892414,0.314332,0.058937,0.432781,0.313989,0.827128,0.474002,0.122987,0.801744,0.888005,0.188060,0.800011


In [35]:
integer_cols

['feat_2',
 'feat_10',
 'feat_39',
 'feat_63',
 'feat_65',
 'feat_78',
 'feat_83',
 'feat_87',
 'feat_101',
 'feat_110',
 'feat_117',
 'feat_121',
 'feat_124',
 'feat_142',
 'feat_145',
 'feat_179',
 'feat_196',
 'feat_202',
 'feat_213',
 'feat_234']

In [36]:
xtrain[integer_cols].head(5)

,feat_2,feat_10,feat_39,feat_63,feat_65,feat_78,feat_83,feat_87,feat_101,feat_110,feat_117,feat_121,feat_124,feat_142,feat_145,feat_179,feat_196,feat_202,feat_213,feat_234
9254,48.0,9.0,31.0,41.0,30.0,36.0,17.0,35.0,14.0,32.0,2.0,9.0,46.0,22.0,4.0,37.0,26.0,32.0,22.0,12.0
1561,1.0,24.0,13.0,15.0,46.0,21.0,43.0,8.0,10.0,30.0,41.0,36.0,46.0,45.0,13.0,33.0,12.0,26.0,37.0,48.0
1670,25.0,20.0,25.0,7.0,41.0,17.0,23.0,6.0,12.0,0.0,35.0,5.0,25.0,16.0,31.0,29.0,8.0,28.0,13.0,29.0
6087,49.0,24.0,1.0,30.0,3.0,12.0,28.0,12.0,44.0,38.0,28.0,42.0,38.0,2.0,25.0,38.0,38.0,45.0,29.0,45.0
6669,42.0,8.0,1.0,47.0,46.0,43.0,10.0,48.0,46.0,40.0,38.0,7.0,30.0,42.0,18.0,15.0,12.0,20.0,10.0,20.0


In [37]:
float_cols

['feat_1',
 'feat_3',
 'feat_4',
 'feat_5',
 'feat_6',
 'feat_7',
 'feat_8',
 'feat_9',
 'feat_11',
 'feat_12',
 'feat_13',
 'feat_14',
 'feat_16',
 'feat_17',
 'feat_18',
 'feat_19',
 'feat_20',
 'feat_21',
 'feat_22',
 'feat_23',
 'feat_24',
 'feat_25',
 'feat_26',
 'feat_27',
 'feat_28',
 'feat_29',
 'feat_30',
 'feat_31',
 'feat_32',
 'feat_33',
 'feat_34',
 'feat_35',
 'feat_36',
 'feat_37',
 'feat_38',
 'feat_40',
 'feat_41',
 'feat_42',
 'feat_43',
 'feat_44',
 'feat_45',
 'feat_46',
 'feat_47',
 'feat_48',
 'feat_49',
 'feat_50',
 'feat_51',
 'feat_52',
 'feat_54',
 'feat_55',
 'feat_56',
 'feat_57',
 'feat_58',
 'feat_59',
 'feat_60',
 'feat_61',
 'feat_62',
 'feat_66',
 'feat_67',
 'feat_68',
 'feat_69',
 'feat_70',
 'feat_72',
 'feat_73',
 'feat_74',
 'feat_75',
 'feat_76',
 'feat_79',
 'feat_80',
 'feat_81',
 'feat_82',
 'feat_84',
 'feat_85',
 'feat_86',
 'feat_88',
 'feat_89',
 'feat_90',
 'feat_91',
 'feat_92',
 'feat_93',
 'feat_94',
 'feat_95',
 'feat_96',
 'feat_97',


In [38]:
xtrain[float_cols].head(5)

,feat_1,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_11,feat_12,...,feat_262,feat_264,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272
9254,4.718762,-0.333381,0.552183,-2.055296,0.523343,2.648409,0.033871,-3.113663,2.245750,-0.090536,...,-3.889656,1.026469,-0.796721,-0.825554,3.861022,1.094665,-0.477780,1.743508,2.958706,-0.595488
1561,1.889255,-0.769640,0.262129,-3.381735,-0.553877,0.980511,-0.112648,2.045847,-1.009131,-1.997801,...,0.310131,3.033212,-1.943146,0.178740,-0.208586,-0.465185,-0.928990,-4.696894,-1.224834,-1.084782
1670,-0.196801,0.914971,2.430244,2.341807,-0.313955,0.736780,1.737302,-0.550688,1.151635,0.273861,...,3.347998,-1.246814,-0.395034,3.019405,-0.571578,-0.020279,-3.307744,-2.621505,0.298648,-0.545975
6087,3.357225,-1.565593,3.500475,0.165899,-3.571130,-0.369367,0.261278,-1.065827,1.194071,1.948701,...,-1.419828,-0.792512,0.273568,-0.566783,0.931091,4.472044,1.066220,-0.616771,0.907830,2.557248
6669,-1.772990,0.715467,-2.924228,1.646592,-1.232597,-1.122442,1.777882,0.239229,3.351670,-3.158781,...,-1.202089,0.497525,-1.535655,1.064860,0.944476,1.642417,-2.589689,0.690000,-0.287031,-1.575887


In [39]:
binary_cols

['feat_229', 'feat_250', 'feat_263']

In [40]:
xtrain[binary_cols].head(5)

,feat_229,feat_250,feat_263
9254,0.0,1.0,1.0
1561,0.0,1.0,0.0
1670,1.0,0.0,0.0
6087,0.0,1.0,0.0
6669,1.0,0.0,0.0


In [41]:
len([*integer_cols, *float_cols, *bounded_01_cols, *binary_cols]) == len(numeric_cols)

True

In [42]:
print(f"Integer columns: {len(integer_cols)}")
print(f"Float columns: {len(float_cols)}")
print(f"Bounded [0,1] columns: {len(bounded_01_cols)}")
print(f"Binary columns: {len(binary_cols)}")

Integer columns: 20
Float columns: 230
Bounded [0,1] columns: 20
Binary columns: 3


In [43]:
# change binary columnns to integer 64 type
xtrain[binary_cols] = xtrain[binary_cols].astype('int64')
xtest[binary_cols] = xtest[binary_cols].astype('int64')

In [44]:
xtrain = xtrain[integer_cols + bounded_01_cols + binary_cols]
xtest = xtest[integer_cols + bounded_01_cols + binary_cols]

In [45]:
xtrain[binary_cols].head(5)

,feat_229,feat_250,feat_263
9254,0,1,1
1561,0,1,0
1670,1,0,0
6087,0,1,0
6669,1,0,0


In [47]:
column_types = {
    'binary': binary_cols,
    'integer': integer_cols,
    'bounded_01': bounded_01_cols,
    'float': float_cols
}

In [48]:
from scipy import stats
outlier_columns = []
for col in [*integer_cols, *bounded_01_cols]:
    z = np.abs(stats.zscore(xtrain[col]))
    outliers = np.count_nonzero(z > 3)  # 3 standard deviations
    if outliers > 0:
        print(f"Column: {col}, Outliers: {outliers}") 
        outlier_columns.append(col)
print(outlier_columns)
print(f"Total columns with outliers: {len(outlier_columns)}")

[]
Total columns with outliers: 0


## 5. Feature Selection via Random Forest Importance

With 270+ features, many carry little signal. A Random Forest is fit on the full feature set purely to read off its **impurity-based feature importances**, and the top **150** features are kept for the modelling stage. This trims noise, speeds up training and reduces over-fitting.

> **Note on transforms.** Outliers were checked with z-scores (below) and options such as 1st/99th-percentile capping and log-transform + `StandardScaler` (via a `ColumnTransformer`) were explored. They were ultimately left out: the final models are tree-based and therefore invariant to monotonic feature scaling and robust to outliers, so the extra complexity earned no gain.

In [52]:
from sklearn.ensemble import RandomForestRegressor

feat_model = RandomForestRegressor(verbose=1, n_jobs=-1, random_state=42)

feat_model.fit(xtrain,ytrain)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    1.7s finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [53]:
# extract feature importance
feat_imp = pd.DataFrame({
    'Features': bounded_01_cols + integer_cols + binary_cols,
    'Importance': feat_model.feature_importances_
}) 

feat_imp = feat_imp.sort_values(by='Importance', ascending=False)

In [55]:
# SAMPLE oNLY 150 BEST FEATURES
best_features = feat_imp.sort_values(by='Importance', ascending=False).head(150)['Features'].values
xtrain = pd.DataFrame(xtrain, columns=bounded_01_cols + integer_cols + binary_cols)[best_features]
xtest = pd.DataFrame(xtest, columns=bounded_01_cols + integer_cols + binary_cols)[best_features]

In [56]:
feat_imp.sort_values(by='Importance',ascending=False)

,Features,Importance
21,feat_10,0.058005
34,feat_145,0.057148
29,feat_110,0.048331
27,feat_87,0.039766
20,feat_2,0.036839
24,feat_65,0.031880
38,feat_213,0.029782
1,feat_15,0.026925
30,feat_117,0.025854
26,feat_83,0.024952


## 6. Model Training & Comparison

Five regressors are trained on the selected features and compared by their **R² on the hold-out test set** (with train R² reported alongside to gauge over-fitting). Tree-based gradient-boosting models are the natural first choice here: they handle mixed feature scales without normalisation and capture non-linear interactions automatically.

### 6.1 · XGBoost Regressor

Gradient-boosted trees with a shallow depth and a low learning rate over many estimators — a strong, well-regularised baseline.

In [58]:
from xgboost import XGBRegressor


xgb_params = {'random_state': 42,
                'n_jobs': -1,
                "verbosity": 2,
                "n_estimators": 1000,
                "learning_rate": 0.06,
                "max_depth": 4,
                "booster": "gbtree",
                "tree_method": "hist"
}


model = XGBRegressor(**xgb_params)

model.fit(xtrain,ytrain)

[11:12:53] INFO: /Users/runner/work/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (8000, 43, 344000).


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,'gbtree'
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diab

In [59]:
model.score(xtrain,ytrain)

0.9676971473902034

In [60]:
model.score(xtest,ytest)

0.8391346699948539

### 6.2 · CatBoost Regressor

CatBoost with **ordered boosting**, which reduces the prediction shift that standard boosting suffers from and often generalises better out of the box.

In [61]:
# Ordered boosting with CatBoost
from catboost import CatBoostRegressor

catboost_params = {'random_state': 42,
                    'iterations': 1500,
                    'learning_rate': 0.05,
                    'depth': 6,
                    'loss_function': 'RMSE',
                    'verbose': 100,
                    'boosting_type': 'Ordered',
                    }
cat_model = CatBoostRegressor(**catboost_params)
cat_model.fit(xtrain,ytrain)

0:	learn: 0.2357045	total: 62.5ms	remaining: 1m 33s
100:	learn: 0.1642589	total: 443ms	remaining: 6.14s
200:	learn: 0.1198956	total: 904ms	remaining: 5.84s
300:	learn: 0.0953834	total: 1.4s	remaining: 5.56s
400:	learn: 0.0875023	total: 2.05s	remaining: 5.62s
500:	learn: 0.0765102	total: 2.47s	remaining: 4.93s
600:	learn: 0.0681558	total: 2.85s	remaining: 4.26s
700:	learn: 0.0625923	total: 3.2s	remaining: 3.64s
800:	learn: 0.0581904	total: 3.58s	remaining: 3.13s
900:	learn: 0.0562001	total: 3.95s	remaining: 2.63s
1000:	learn: 0.0545012	total: 4.32s	remaining: 2.15s
1100:	learn: 0.0531657	total: 4.71s	remaining: 1.71s
1200:	learn: 0.0497397	total: 5.08s	remaining: 1.26s
1300:	learn: 0.0487770	total: 5.47s	remaining: 837ms
1400:	learn: 0.0478041	total: 5.89s	remaining: 416ms
1499:	learn: 0.0460926	total: 6.33s	remaining: 0us


In [62]:
cat_model.score(xtrain,ytrain)

np.float64(0.9620163176127882)

In [63]:
cat_model.score(xtest,ytest)

np.float64(0.9117786715057753)

In [64]:
cat_model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,feat_110,27.834179
1,feat_10,26.212874
2,feat_117,23.227004
3,feat_0,3.427944
4,feat_15,2.970104
5,feat_126,2.909395
6,feat_204,2.837516
7,feat_162,2.221446
8,feat_71,1.035961
9,feat_222,0.784352


In [65]:
y_pred = cat_model.predict(xtest)
np.sqrt(mean_squared_error(ytest, y_pred))

np.float64(0.07204496082010324)

### 6.3 · LightGBM Regressor

A fast, histogram-based gradient-boosting implementation used here as a cross-check on the other boosters.

In [66]:
# dont use previous models, start fresh
from lightgbm import LGBMRegressor
lgbm_params = {'random_state': 42,
                'n_jobs': -1,
                'max_depth': 6
                }
lgbm_model = LGBMRegressor(**lgbm_params)
lgbm_model.fit(xtrain,ytrain)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000636 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6106
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 43
[LightGBM] [Info] Start training from score 0.719608
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [67]:
lgbm_model.score(xtrain,ytrain)

0.8291915803684968

In [68]:
lgbm_model.score(xtest,ytest)

0.6316508660193287

### 6.4 · Random Forest Regressor

A bagged-tree ensemble — lower variance than a single tree and a useful non-boosting point of comparison.

In [69]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42,n_estimators=500, n_jobs=-1, verbose=1)

model.fit(xtrain,ytrain)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    7.3s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    8.5s finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [70]:
model.score(xtrain,ytrain)

[Parallel(n_jobs=11)]: Using backend ThreadingBackend with 11 concurrent workers.
[Parallel(n_jobs=11)]: Done  28 tasks      | elapsed:    0.0s
[Parallel(n_jobs=11)]: Done 178 tasks      | elapsed:    0.0s
[Parallel(n_jobs=11)]: Done 428 tasks      | elapsed:    0.1s
[Parallel(n_jobs=11)]: Done 500 out of 500 | elapsed:    0.1s finished


0.8816154160359163

In [71]:
model.score(xtest,ytest)

[Parallel(n_jobs=11)]: Using backend ThreadingBackend with 11 concurrent workers.
[Parallel(n_jobs=11)]: Done  28 tasks      | elapsed:    0.0s
[Parallel(n_jobs=11)]: Done 178 tasks      | elapsed:    0.0s
[Parallel(n_jobs=11)]: Done 428 tasks      | elapsed:    0.0s
[Parallel(n_jobs=11)]: Done 500 out of 500 | elapsed:    0.0s finished


0.1351205118147304

### 6.5 · Linear Regression (baseline)

An ordinary least-squares baseline. If the relationship were largely linear it would be competitive; the gap to the boosted models quantifies how much non-linearity the tree ensembles are capturing.

In [72]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(xtrain,ytrain)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [73]:
lr.score(xtrain,ytrain)

0.08789305192558705

In [74]:
lr.score(xtest,ytest)

0.07039165751071441

## 7. Final Evaluation & Submission

CatBoost is the strongest model on the hold-out set, so it is used to score the unlabeled evaluation data. The evaluation features are put through the **exact same** preprocessing (dtype fixes + feature selection) as the training data before prediction.

### 7.1 · Load the evaluation features

In [75]:
xeval = pd.read_csv("problem_41/EVAL_41.csv")

In [76]:
xeval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 273 entries, feat_0 to feat_272
dtypes: float64(273)
memory usage: 20.8 MB


### 7.2 · Preprocessing

Apply the identical dtype handling and column selection used on the training set so the model sees features in the shape it expects.

#### Handling data types

In [78]:
# change binary columnns to integer 64 type
xeval[binary_cols] = xeval[binary_cols].astype('int64')

#### Feature selection

In [79]:
xeval = xeval[integer_cols + bounded_01_cols + binary_cols]

### 7.3 · Predict & write the submission

Score the evaluation set with the trained CatBoost model and save the predictions to `EVAL_target01_41.csv`.

In [81]:
targets = cat_model.predict(xeval)

In [82]:
# Saving to a CSV file with target01 column and file name EVAL_target01_41.csv
submission = pd.DataFrame({
    'target01': targets
})

In [83]:
submission.to_csv("EVAL_target01_41.csv", index=False)

## 8. Conclusion

- Every feature was typed from its raw values and the field was narrowed to the **150 most important** features via Random Forest importances.
- Five regressors were compared on an identical hold-out set. The gradient-boosting models clearly outperformed the linear baseline, confirming meaningful non-linear structure in the data.
- **CatBoost (ordered boosting)** gave the best hold-out performance and was chosen as the final model.
- Predictions for the evaluation set are saved to **`EVAL_target01_41.csv`**.

*Possible next steps: hyper-parameter search (Optuna), cross-validated model selection, and blending the top boosters into an ensemble.*